In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data.dataset import NucleotideDataset, MLMPretrainingDataset

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 2 
MODEL_DIM = 32
NUM_HEADS = 8
NUM_LAYERS = 4

MASK_PROB = 0.15
RAND_PROB = 0.1

In [ ]:
train_data_raw = torch.load(
    r"..\data\processed\human_nontata_promoters\train\nucleotide_dataset.pt"
)
test_data_raw = torch.load(
    r"..\data\processed\human_nontata_promoters\test\nucleotide_dataset.pt"
)

In [ ]:
train_data = MLMPretrainingDataset(
    train_data_raw,
    mask_prob=MASK_PROB,
    rand_prob=RAND_PROB,
    max_length=MAX_LENGTH,
) 

test_data = MLMPretrainingDataset(
    test_data_raw,
    mask_prob=MASK_PROB,
    rand_prob=RAND_PROB,
    max_length=MAX_LENGTH,
)

In [ ]:
dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

print(f"Batch sample: {next(iter(dataloader))[0].shape}")

In [ ]:
from src.models.transformer import Transformer
import torch.nn.functional as F

In [ ]:
encoder = Transformer(
    num_unique_tokens=9,  # 4 nucleotides + 5 special tokens
    model_dim=MODEL_DIM,
    num_heads=NUM_HEADS,
    max_len=MAX_LENGTH
)

In [ ]:
seq_input = next(iter(dataloader))
print(seq_input[0][0])
print(seq_input[1][0])

In [ ]:
class Model(nn.Module):
    def __init__(self, transformer):
        super(Model, self).__init__()
        self.embedder = nn.Embedding(9, MODEL_DIM)  # 9 unique tokens
        self.transformer = transformer
        self.output_projection = nn.Linear(MODEL_DIM, 9)  # Output layer for token prediction
        self.output_projection.weight = self.embedder.weight

    def forward(self, x, mask=None):

        embedded_x = self.embedder(x)
        transformer_output = self.transformer(embedded_x, mask=None)
        logits = self.output_projection(transformer_output)

        return logits

In [ ]:
transformer = Model(transformer=encoder)

In [ ]:
seq_input[0]

In [ ]:
transformer_output = transformer(seq_input[0], seq_input[1])

In [ ]:
transformer_output.view(-1, 9).shape

In [ ]:
seq_input[2].view(-1).shape

In [ ]:
loss = F.cross_entropy(
    transformer_output.view(-1, 9),
    seq_input[2].view(-1)
)

In [ ]:
loss

In [ ]:
transformer.to(torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. Using CPU.")

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (PyTorch build):", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))